In [2]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("../Datasets/processed/UHPC_dataset/initial_cleaned.csv", index_col=0)
df = df.reset_index(drop=True)
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (2072, 43)


,cement,cement_type,cement_grade,silica_fume,fly_ash,fly_ash_type,limestone_powder,quartz_powder,glass_powder,rice_husk_ash,...,fiber2_tensile_strength,fiber2_youngs_modulus,water,sp_type,sp_amount,curing_method,curing_temp,curing_humidity,curing_pressure,cs_28d
0,839.0,OPC_I,NaN,104.0,52.0,class F,0.0,0.0,0.0,0.0,...,NaN,NaN,147.0,PCE_SP,59.33,Standard Curing,NaN,NaN,NaN,132.0
1,839.0,OPC_I,NaN,104.0,26.0,class F,0.0,0.0,0.0,0.0,...,NaN,NaN,147.0,PCE_SP,59.33,Standard Curing,NaN,NaN,NaN,122.5
2,839.0,OPC_I,NaN,104.0,0.0,NaN,0.0,0.0,0.0,0.0,...,NaN,NaN,147.0,PCE_SP,62.15,Standard Curing,NaN,NaN,NaN,116.0
3,839.0,OPC_I,NaN,104.0,52.0,class F,0.0,0.0,0.0,0.0,...,NaN,NaN,147.0,PCE_SP,64.98,Standard Curing,NaN,NaN,NaN,134.0
4,839.0,OPC_I,NaN,104.0,26.0,class F,0.0,0.0,0.0,0.0,...,NaN,NaN,147.0,PCE_SP,64.98,Standard Curing,NaN,NaN,NaN,131.5


# Semantic Missingness Analysis

Analyzing and recoding missing values in UHPC dataset, distinguishing between structural zeros (not applicable) and actual missing data (unknown).

In [3]:
## Data Filtering Utility

def drop_by_threshold(df, threshold_pct):
    """Drop columns where missing values exceed threshold percentage."""
    threshold = threshold_pct * len(df)
    dropped = df.columns[df.isna().sum() > threshold].tolist()
    print(f"Dropped columns ({int(threshold_pct*100)}% threshold): {dropped}")
    return df.loc[:, df.isna().sum() <= threshold]

In [4]:
# 50% missing data threshold
df_raw_50 = df.copy()
df_raw_50 = drop_by_threshold(df_raw_50, 0.5)
df_raw_50 = df_raw_50.reset_index(drop=True)
print(f"50% threshold shape: {df_raw_50.shape}")

Dropped columns (50% threshold): ['fly_ash_type', 'slag_type', 'filler_type', 'fiber1_tensile_strength', 'fiber1_youngs_modulus', 'fiber2_type', 'fiber2_amount', 'fiber2_length', 'fiber2_diameter', 'fiber2_tensile_strength', 'fiber2_youngs_modulus', 'curing_humidity', 'curing_pressure']
50% threshold shape: (2072, 30)


## Create Datasets with Different Missing Data Thresholds

In [5]:
# 20% missing data threshold
df_raw_20 = df.copy()
df_raw_20 = drop_by_threshold(df_raw_20, 0.2)
df_raw_20 = df_raw_20.reset_index(drop=True)
print(f"20% threshold shape: {df_raw_20.shape}")

Dropped columns (20% threshold): ['cement_grade', 'fly_ash_type', 'slag_type', 'filler_type', 'fiber1_type', 'fiber1_amount', 'fiber1_length', 'fiber1_diameter', 'fiber1_tensile_strength', 'fiber1_youngs_modulus', 'fiber2_type', 'fiber2_amount', 'fiber2_length', 'fiber2_diameter', 'fiber2_tensile_strength', 'fiber2_youngs_modulus', 'curing_humidity', 'curing_pressure']
20% threshold shape: (2072, 25)


In [6]:
## Semantic Missingness Recoding

# Define material amount/type pairs for semantic analysis
amount_type_twins = [
    ['fly_ash', 'fly_ash_type'],
    ['slag', 'slag_type'],
    ['filler', 'filler_type'],
    ['sand', 'sand_type'],
    ['fiber1_amount', 'fiber1_type'],
    ['fiber2_amount', 'fiber2_type'],
    ['sp_amount', 'sp_type']
]

In [7]:
def recode_semantic_missingness(df, amount_col, type_col):
    """Recode missing values based on material amount.
    
    Rules:
    - amount > 0 & type NaN → 'unknown_type' (material used, type unknown)
    - amount NaN/0 & type NaN → 'not_applicable' (material not used)
    """
    # Material used but type unknown
    mask1 = (df[amount_col] > 0) & df[type_col].isna()
    df.loc[mask1, type_col] = 'unknown_type'

    # Material not used (amount 0 or NaN) and type NaN
    mask2 = (df[amount_col].isna() | (df[amount_col] == 0)) & df[type_col].isna()
    df.loc[mask2, type_col] = 'not_applicable'
    df.loc[mask2, amount_col] = 0

    return df

# Apply semantic recoding
df_semantic_recoded = df.copy()

for amount_col, type_col in amount_type_twins:
    before_amount = df[amount_col].isna().sum()
    before_type = df[type_col].isna().sum()
    
    recode_semantic_missingness(df_semantic_recoded, amount_col, type_col)
    
    print(f"\n{amount_col} / {type_col}")
    print(f"  NaN amount:      {before_amount} → {df[amount_col].isna().sum()}  (true reporting gap)")
    print(f"  NaN type:        {before_type} → {df[type_col].isna().sum()}  (true reporting gap)")
    print(f"  unknown_type:    {(df[type_col] == 'unknown_type').sum()}  (recoded)")
    print(f"  not_applicable:  {(df[type_col] == 'not_applicable').sum()}  (recoded)")


fly_ash / fly_ash_type
  NaN amount:      0 → 0  (true reporting gap)
  NaN type:        1648 → 1648  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

slag / slag_type
  NaN amount:      0 → 0  (true reporting gap)
  NaN type:        1980 → 1980  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

filler / filler_type
  NaN amount:      0 → 0  (true reporting gap)
  NaN type:        1712 → 1712  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

sand / sand_type
  NaN amount:      0 → 0  (true reporting gap)
  NaN type:        81 → 81  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

fiber1_amount / fiber1_type
  NaN amount:      669 → 669  (true reporting gap)
  NaN type:        669 → 669  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

fiber2_amount / fiber2_type
  NaN amount:      1954 → 1954  (true repo

In [8]:
# Recoded + 50% missing data threshold
df_semantic_recod_50 = df_semantic_recoded.copy()
df_semantic_recod_50 = drop_by_threshold(df_semantic_recod_50, 0.5)
print(f"Recoded + 50% threshold shape: {df_semantic_recod_50.shape}")

Dropped columns (50% threshold): ['fiber1_tensile_strength', 'fiber1_youngs_modulus', 'fiber2_length', 'fiber2_diameter', 'fiber2_tensile_strength', 'fiber2_youngs_modulus', 'curing_humidity', 'curing_pressure']
Recoded + 50% threshold shape: (2072, 35)


## Create Final Datasets (Recoded + Threshold)

In [9]:
# Recoded + 20% missing data threshold
df_semantic_recod_20 = df_semantic_recoded.copy()
df_semantic_recod_20 = drop_by_threshold(df_semantic_recod_20, 0.2)
print(f"Recoded + 20% threshold shape: {df_semantic_recod_20.shape}")


Dropped columns (20% threshold): ['cement_grade', 'fiber1_length', 'fiber1_diameter', 'fiber1_tensile_strength', 'fiber1_youngs_modulus', 'fiber2_length', 'fiber2_diameter', 'fiber2_tensile_strength', 'fiber2_youngs_modulus', 'curing_humidity', 'curing_pressure']
Recoded + 20% threshold shape: (2072, 32)


In [10]:
df_semantic_recod_50.to_csv("../Datasets/processed//uhpc_dataset/semantic_recoding_features_50.csv", index=False)
df_semantic_recod_20.to_csv("../Datasets/processed//uhpc_dataset/semantic_recoding_features_20.csv", index=False)
df_raw_50.to_csv("../Datasets/processed//uhpc_dataset/raw_dropped_features_50.csv", index=False)
df_raw_20.to_csv("../Datasets/processed//uhpc_dataset/raw_dropped_features_20.csv", index=False)

In [11]:
df_semantic_recod_20.isnull().sum()[df_semantic_recod_20.isnull().sum() > 0]

sand_max_size    276
sp_amount          8
curing_temp      157
dtype: int64

In [12]:
df_semantic_recod_20['sand'][df_semantic_recod_20['sand_max_size'].isna()].describe()

count     276.000000
mean      816.853913
std       397.325703
min         0.000000
25%       653.000000
50%       973.350000
75%      1075.000000
max      1756.480000
Name: sand, dtype: float64

In [13]:
num_cols = df_semantic_recod_50.select_dtypes(include='number').columns

Q1 = df_semantic_recod_50[num_cols].quantile(0.25)
Q3 = df_semantic_recod_50[num_cols].quantile(0.75)
IQR = Q3 - Q1

outlier_counts = ((df_semantic_recod_50[num_cols] < (Q1 - 1.5 * IQR)) | (df_semantic_recod_50[num_cols] > (Q3 + 1.5 * IQR))).sum()
outlier_pct = (outlier_counts / len(df_semantic_recod_50) * 100).round(1)

summary = pd.DataFrame({'outlier_count': outlier_counts, 'outlier_%': outlier_pct})
print(summary[summary['outlier_count'] > 0].sort_values('outlier_%', ascending=False))

                  outlier_count  outlier_%
fiber1_length               523       25.2
quartz_powder               514       24.8
fly_ash                     435       21.0
filler                      360       17.4
ggbfs                       235       11.3
fiber1_diameter             215       10.4
sand                        190        9.2
glass_powder                189        9.1
water                       129        6.2
limestone_powder            126        6.1
fiber2_amount               118        5.7
slag                         96        4.6
nano_sio2                    95        4.6
curing_temp                  76        3.7
sand_max_size                77        3.7
metakaolin                   73        3.5
cement                       53        2.6
cs_28d                       46        2.2
nano_caco3                   44        2.1
rice_husk_ash                28        1.4
sp_amount                    27        1.3
fiber1_amount                21        1.0
silica_fume

In [14]:
Q1 = df_semantic_recod_50['cs_28d'].quantile(0.25)
Q3 = df_semantic_recod_50['cs_28d'].quantile(0.75)
IQR = Q3 - Q1
mask = (df_semantic_recod_50['cs_28d'] < Q1 - 1.5*IQR) | (df_semantic_recod_50['cs_28d'] > Q3 + 1.5*IQR)
print(df_semantic_recod_50.loc[mask, 'cs_28d'].describe())

count     46.000000
mean     258.869565
std       14.035841
min      239.000000
25%      248.250000
50%      257.000000
75%      268.000000
max      298.000000
Name: cs_28d, dtype: float64


In [15]:
df_semantic_recod_50.select_dtypes(include='object').shape[1]

C:\Users\shoai\AppData\Local\Temp\ipykernel_3060\2735213181.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_semantic_recod_50.select_dtypes(include='object').shape[1]


9